In [ ]:
import numpy as np

def inside_kidney(x, y, eps=1e-15):
    return (x*x + y*y)**2 <= (x**3 + y**3) + eps

def inside_disc(x, y, eps=1e-15):
    return (x - 0.25)**2 + (y - 0.25)**2 <= 0.125 + eps

def remaining_indicator(x, y, eps=1e-15):
    return inside_kidney(x, y, eps) & (~inside_disc(x, y, eps))

def estimate_bbox(mask_func, search_box=(-1.0, 1.5, -1.0, 1.5), n=2000, margin_steps=2):

    xmin, xmax, ymin, ymax = search_box
    xs = np.linspace(xmin, xmax, n)
    ys = np.linspace(ymin, ymax, n)
    X, Y = np.meshgrid(xs, ys, indexing="xy")

    mask = mask_func(X, Y)
    if not np.any(mask):
        raise RuntimeError("No points found. Enlarge search_box.")

    x_in = X[mask]; y_in = Y[mask]
    x0, x1 = x_in.min(), x_in.max()
    y0, y1 = y_in.min(), y_in.max()

    dx = (xmax - xmin) / (n - 1)
    dy = (ymax - ymin) / (n - 1)

    return (x0 - margin_steps*dx, x1 + margin_steps*dx,
            y0 - margin_steps*dy, y1 + margin_steps*dy)

In [ ]:
def area_rectangle_midpoint(n, bbox):
    xmin, xmax, ymin, ymax = bbox
    dx = (xmax - xmin) / n
    dy = (ymax - ymin) / n

    xs = xmin + (np.arange(n) + 0.5) * dx
    ys = ymin + (np.arange(n) + 0.5) * dy
    X, Y = np.meshgrid(xs, ys, indexing="xy")

    inside = remaining_indicator(X, Y)
    return inside.sum() * dx * dy

def area_trapezoid_2d(n, bbox):
    xmin, xmax, ymin, ymax = bbox
    dx = (xmax - xmin) / n
    dy = (ymax - ymin) / n

    xs = xmin + np.arange(n + 1) * dx
    ys = ymin + np.arange(n + 1) * dy
    X, Y = np.meshgrid(xs, ys, indexing="xy")

    g = remaining_indicator(X, Y).astype(float)

    w = np.ones_like(g)
    w[0, :] *= 0.5
    w[-1, :] *= 0.5
    w[:, 0] *= 0.5
    w[:, -1] *= 0.5

    return (g * w).sum() * dx * dy

def area_monte_carlo(M, bbox, seed=42):
    xmin, xmax, ymin, ymax = bbox
    rng = np.random.default_rng(seed)

    xs = rng.uniform(xmin, xmax, M)
    ys = rng.uniform(ymin, ymax, M)
    inside = remaining_indicator(xs, ys)

    return inside.mean() * (xmax - xmin) * (ymax - ymin)

In [ ]:
if __name__ == "__main__":
    bbox = estimate_bbox(
        mask_func=lambda X, Y: remaining_indicator(X, Y),
        search_box=(-1.0, 1.5, -1.0, 1.5),
        n=2400,
        margin_steps=2
    )
    print("Estimated bbox (remaining region):", bbox)

    n_grid = 1200
    M_mc = 3_000_000

    A_rect = area_rectangle_midpoint(n_grid, bbox)
    A_trap = area_trapezoid_2d(n_grid, bbox)
    A_mc   = area_monte_carlo(M_mc, bbox, seed=42)

    print("\nRemaining Kidney Area Estimates")
    print(f"Rectangle (midpoint), n={n_grid}: {A_rect:.10f}")
    print(f"Trapezoid (2D),      n={n_grid}: {A_trap:.10f}")
    print(f"Monte Carlo,   M={M_mc}: {A_mc:.10f}")

    print("\nTo 4 significant digits:")
    print(f"Rectangle: {A_rect:.4g}")
    print(f"Trapezoid: {A_trap:.4g}")
    print(f"MC:        {A_mc:.4g}")

Estimated bbox (remaining region): (np.float64(-0.2840766986244268), np.float64(1.0018757815756565), np.float64(-0.2840766986244268), np.float64(1.0018757815756565))

Remaining Kidney Area Estimates
Rectangle (midpoint), n=1200: 0.5890546783
Trapezoid (2D),      n=1200: 0.5889949623
Monte Carlo,   M=3000000: 0.5886400655

To 4 significant digits:
Rectangle: 0.5891
Trapezoid: 0.589
MC:        0.5886


In [ ]:
import numpy as np

def solve_and_verify(N=126, seed=123):
    rng = np.random.default_rng(seed)

    A = rng.uniform(-1.0, 1.0, size=(N, N))
    b = np.ones(N)

    # Solve Ax=b
    x = np.linalg.solve(A, b)

    # L1 residual norm ||Ax - b||_1
    r = A @ x - b
    res_l1 = np.linalg.norm(r, ord=1)

    return A, b, x, res_l1

if __name__ == "__main__":
    N = 126
    A, b, x, res_l1 = solve_and_verify(N=N, seed=123)

    print(f"N = {N}")
    print(f"L1 residual norm ||Ax-b||_1 = {res_l1:.6e}")
    print(f"First 5 entries of x: {x[:5]}")

N = 126
L1 residual norm ||Ax-b||_1 = 2.199463e-12
First 5 entries of x: [-3.88450332  1.33284807  0.98079958 -0.21308766 -3.18064258]
